# CAPM Portfolio Optimizer — Bank Nifty
> **End-to-end pipeline:** Data Ingestion → Financial Modelling → Portfolio Optimization → Delta Tables for Power BI  
> **Platform:** Microsoft Fabric (Synapse PySpark Notebook)  
> **Methodology:** Custom Noise-Adjusted CAGR · CAPM · Sharpe Ratio · Correlation Filtering · Minimum-Variance Weighting


In [0]:
!pip install yfinance
import pandas as pd
import yfinance as yf
import numpy as np

## 1 · Input Configuration

| Parameter | Description |
|---|---|
| `Stocks_List` | NSE ticker symbols to analyse (suffix `.NS` required) |
| `period` | Historical data window in years |
| `rf` | Risk-free rate — use your opportunity cost (e.g. current FD / G-Sec rate) |
| `Market` | Benchmark index for Beta calculation — choose the index that best represents your stocks |

> **Tip:** If you change the number of stocks, adjust your Power BI visuals accordingly.


In [0]:
Stocks_List = [
    'AUBANK.NS',    'AXISBANK.NS',  'BANDHANBNK.NS',
    'BANKBARODA.NS','CANBK.NS',     'FEDERALBNK.NS',
    'HDFCBANK.NS',  'ICICIBANK.NS', 'IDFCFIRSTB.NS',
    'INDUSINDBK.NS','KOTAKBANK.NS', 'PNB.NS',
    'SBIN.NS',      'UNIONBANK.NS'
]
period = 5
rf     = 0.0678
Market = '^NSEBANK'


## 2 · Data Ingestion — Silver Table

Fetches OHLCV data from **yfinance** for all tickers and the benchmark index.  
The loop is required here because `yf.Ticker` makes individual HTTP calls per symbol.

> You can swap yfinance for any internal/enterprise data API without changing anything downstream — just ensure the output `df3` has the same schema: `Date | Symbol | Stock_Close`.


In [0]:
Combined_Stocks = []
for banks in Stocks_List:
    df1 = yf.Ticker(banks)
    df2 = df1.history(period=f"{period}y")
    df2 = df2.reset_index()
    df2['Date']        = df2['Date'].dt.tz_localize(None)
    df2['Symbol']      = banks
    df2                = df2[['Date', 'Symbol', 'Close']].rename(columns={'Close': 'Stock_Close'})
    df2['Stock_Close'] = df2['Stock_Close'].round(2)
    df2['Symbol']      = df2['Symbol'].str[:-3]
    Combined_Stocks.append(df2)

df3 = pd.concat(Combined_Stocks)


## 3 · Daily Stock Returns

Calculates daily percentage return per stock using `pct_change()` grouped by symbol.  
Rows with `NaN` (first row per stock) are dropped.


In [0]:
df = df3.copy()
df['Stock_Return'] = df.groupby('Symbol')['Stock_Close'].pct_change()
df = df.dropna()
df.head()


## 4 · Benchmark (Market) Returns

Fetches Bank Nifty index data and computes daily returns.  
This serves as the **market portfolio** for all CAPM and Beta calculations.


In [0]:
nf = yf.Ticker(Market)
nf = nf.history(period=f"{period}y")
nf = nf.reset_index()
nf['Date']       = nf['Date'].dt.tz_localize(None)
nf               = nf[['Date', 'Close']].rename(columns={'Close': 'nse_close'})
nf['nse_close']  = nf['nse_close'].round(2)
nf['Nse_Return'] = nf['nse_close'].pct_change()
nf               = nf.dropna()
nf.head()


## 5 · Master Working Table (`mf`)

Merges stock returns with market returns on `Date` (inner join — only trading days present in both).  
A `Day_Count` column is added to track how many days ago each record is from the latest date — useful for time-weighting in Power BI.


In [0]:
mf = nf.merge(df, on='Date', how='inner')

max_date        = mf['Date'].max()
mf['Day_Count'] = (max_date - mf['Date']).dt.days
print('Latest trading date:', max_date)

mf = mf[['Date', 'Symbol', 'Stock_Return', 'Nse_Return', 'Day_Count']]
mf.head()


## 6 · CAPM — Beta & Expected Return

**Formula:**  
`E(R) = rf + β × (rm − rf)`

| Variable | Meaning |
|---|---|
| `β (Beta)` | Sensitivity of stock return to market movement. β > 1 = amplifies market moves |
| `rm` | Annualised average market return (daily mean × 252 trading days) |
| `rf` | Risk-free rate — your opportunity cost |
| `E(R)` | Minimum expected return to justify holding this stock over a risk-free asset |

> This is the **textbook CAPM** approach. The custom CAGR calculated later captures actual realised return, which is then compared against this expected return to compute Alpha.


In [0]:
beta = mf.groupby('Symbol').apply(
    lambda x: x['Stock_Return'].cov(x['Nse_Return']) / x['Nse_Return'].var()
)

rm           = nf['Nse_Return'].mean() * 252
capm_return  = rf + (beta * (rm - rf))

capm_report  = pd.DataFrame({
    'Beta'        : beta.round(4),
    'capm_return' : capm_return.round(4)
})
capm_report = capm_report.reset_index()
capm_report


## 7 · Correlation Matrix

Measures how similarly any two stocks move on a daily basis.  

**Why this matters for portfolio construction:**  
Holding two highly correlated stocks (e.g. BANKBARODA and CANBK at ~0.80) provides almost no diversification benefit — you are essentially doubling your exposure to the same risk factor.  
Stocks with correlation < 0.6 are preferred candidates for the final portfolio.


In [0]:
cr  = mf.pivot(index='Date', columns='Symbol', values='Stock_Return')
cr1 = cr.corr().reset_index()
cr1


### Melting the Correlation Matrix for Power BI

Power BI requires a long (melted) format to build a matrix visual — this cell transforms the wide correlation table into `Symbol | bank | correlation` rows.


In [0]:
melt = cr1.melt(id_vars='Symbol', value_name='correlation', var_name='bank')
display(melt)


In [0]:
%sql
create schema if not exists dbo

In [0]:
spark_melt = spark.createDataFrame(melt)
spark_melt.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.V2Melt')


## 8 · Custom Noise-Adjusted CAGR (Actual Return)

### The Problem with Textbook CAGR
Standard CAGR uses a **single closing price** as the start and end point.  
This creates two risks:
- **Endpoint bias:** A crash or spike on that exact day gives a completely distorted picture.
- **Timing luck:** Real investors do not invest at a precise second — money flows in over time.

### The Solution: Mean-Based CAGR
Instead of point-to-point, this model:
1. Takes the **mean price of the first year** (Year 1 Mean) as the effective entry price.
2. Takes the **mean price of the last year** (Year N Mean) as the effective exit price.
3. Applies a **Z-Score outlier filter (±3σ)** on daily returns *before* computing the mean — this removes extreme events like circuit breakers, news spikes, and irrational panic that would otherwise distort the mean.

**Formula:**  
`CAGR = (Year_N_Mean / Year_1_Mean) ^ (1 / (period − 1)) − 1`

> The `(period − 1)` denominator correctly accounts for the time elapsed *between* the midpoints of the two yearly windows, not the full period length.

### Z-Score Optimisation Note
The Z-score is computed on **daily returns** (not price levels) and applied **per symbol** using vectorised `.map()` instead of `apply(axis=1)`.  
This avoids row-by-row Python iteration, which saves significant CUs in Fabric.


In [0]:
cag          = df3[['Date', 'Symbol', 'Stock_Close']]
Days_Required = round(1 / period * cag['Symbol'].count(), 0).astype('int64')
print('Days per year window:', Days_Required)


### Year 1 Mean — Entry Price Baseline


In [0]:
cag1 = cag.sort_values('Date', ascending=True).head(Days_Required).copy()

cag1['Stock_Return'] = cag1.groupby('Symbol')['Stock_Close'].pct_change()
cag1 = cag1.dropna()

Stock_Mean1 = cag1.groupby('Symbol')['Stock_Return'].mean()
Std_Dev1    = cag1.groupby('Symbol')['Stock_Return'].std()

# Vectorised Z-Score — avoids row-by-row apply(), saves CUs
cag1['Z_Score'] = (
    (cag1['Stock_Return'] - cag1['Symbol'].map(Stock_Mean1))
    / cag1['Symbol'].map(Std_Dev1)
)
cag1 = cag1[cag1['Z_Score'].between(-3, 3)]

Stock_Mean_Year1 = (
    cag1.groupby('Symbol')['Stock_Close'].mean()
        .reset_index()
        .rename(columns={'Stock_Close': 'Year1_Mean'})
)
Stock_Mean_Year1


### Year N Mean — Exit Price Baseline


In [0]:
cag2 = cag.sort_values('Date', ascending=False).head(Days_Required).copy()
cag2 = cag2.sort_values('Date', ascending=True)

cag2['Stock_Return'] = cag2.groupby('Symbol')['Stock_Close'].pct_change()
cag2 = cag2.dropna()

Stock_Mean2 = cag2.groupby('Symbol')['Stock_Return'].mean()
Std_Dev2    = cag2.groupby('Symbol')['Stock_Return'].std()

# Vectorised Z-Score — avoids row-by-row apply(), saves CUs
cag2['Z_Score'] = (
    (cag2['Stock_Return'] - cag2['Symbol'].map(Stock_Mean2))
    / cag2['Symbol'].map(Std_Dev2)
)
cag2 = cag2[cag2['Z_Score'].between(-3, 3)]

Stock_Mean_Year5 = (
    cag2.groupby('Symbol')['Stock_Close'].mean()
        .reset_index()
        .rename(columns={'Stock_Close': 'Year5_Mean'})
)
Stock_Mean_Year5.head(3)


In [0]:
cagr_fresh         = Stock_Mean_Year1.merge(Stock_Mean_Year5, on='Symbol')
cagr_fresh['CAGR'] = (cagr_fresh['Year5_Mean'] / cagr_fresh['Year1_Mean']) ** (1 / (period - 1)) - 1
cagr_fresh


## 9 · Risk Profile — Gold Table

Assembles the final analytical table by merging all computed metrics:

| Column | Meaning |
|---|---|
| `Closing_Price` | Latest available price |
| `Day_Return` | Last day's return |
| `Year1_Mean / Year5_Mean` | Noise-adjusted entry/exit price estimates |
| `CAGR` | Custom noise-adjusted actual return |
| `CAPM` | Minimum expected return (textbook) |
| `Beta` | Market sensitivity |
| `Return_SD` | Annualised return volatility (σ × √252) |
| `Alpha` | Excess return over CAPM — the value a stock generates beyond market compensation |
| `Market_SD` | Annualised benchmark volatility |
| `Sharpe_Ratio` | Risk-adjusted return: (CAGR − rf) / σ |
| `rm` | Annualised market return |


In [0]:
# 1. Annualised stock return volatility
SD = df.groupby('Symbol')['Stock_Return'].std() * (252 ** 0.5)
SD = SD.reset_index()

# 2. Latest price and last day return per stock
Price_Today = df3.groupby('Symbol')['Date'].max().reset_index()
Price_Today = Price_Today.merge(df3, on=['Symbol', 'Date'])
Price_Today = Price_Today.rename(columns={'Stock_Close': 'Closing_Price'})

# 3. Assemble Gold table
Risk_Profile = (
    capm_report
    .merge(cagr_fresh,  on='Symbol')
    .merge(SD,          on='Symbol')
    .merge(Price_Today, on='Symbol')
)

Risk_Profile = (
    Risk_Profile[
        ['Symbol', 'Date', 'Closing_Price', 'Year5_Mean', 'Year1_Mean',
         'CAGR', 'capm_return', 'Beta', 'Stock_Return']
    ]
    .rename(columns={'capm_return': 'CAPM', 'Stock_Return': 'Return_SD'})
)

# 4. Alpha — excess return over CAPM
Risk_Profile['Alpha']        = Risk_Profile['CAGR'] - Risk_Profile['CAPM']

# 5. Annualised market volatility
Risk_Profile['Market_SD']    = nf['Nse_Return'].std() * (252 ** 0.5)

# 6. Sharpe Ratio
Risk_Profile['rf']           = rf
Risk_Profile['Sharpe_Ratio'] = (Risk_Profile['CAGR'] - rf) / Risk_Profile['Return_SD']

# 7. Annualised market return
Risk_Profile['rm']           = nf['Nse_Return'].mean() * 252

Risk_Profile


In [0]:
spark_rp = spark.createDataFrame(Risk_Profile)
spark_rp.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.V2_Risk_Report')


## 10 · Portfolio Stock Selection

A multi-step quantitative filter to identify stocks worthy of inclusion in the final portfolio.

| Step | Filter | Rationale |
|---|---|---|
| 1 | Alpha > 0 | Stock must generate returns above what CAPM predicts — otherwise a risk-free instrument is better |
| 2 | Sharpe Ratio > 0.5 | Every unit of risk taken must generate meaningful excess return |
| 3 | Pairwise Correlation < 0.6 | Ensures diversification — eliminates stocks that are too similar to each other |
| 4 | 0.5 ≤ Beta < 1 | Avoids stocks that merely mirror the index (Beta ≈ 1) or are excessively volatile (Beta > 1) |
| 5 | CAPM > rf | The market-implied required return must itself exceed the risk-free rate — basic sanity check |


### Step 1 & 2 — Alpha and Sharpe Filter


In [0]:
Step1 = Risk_Profile[Risk_Profile['Alpha'] > 0]
Step2 = Step1[Step1['Sharpe_Ratio'] > 0.5]
Step2


### Step 3 — Correlation Filter

Re-computes the correlation matrix using only the Step 2 survivors, then keeps only pairs where correlation is below 0.6.


In [0]:
Step3_0 = mf[mf['Symbol'].isin(Step2['Symbol'])]
Step3_1 = Step3_0.pivot(index='Date', columns='Symbol', values='Stock_Return')
Step3_2 = Step3_1.corr().reset_index()
Step3_3 = Step3_2.melt(id_vars='Symbol', value_name='correlation', var_name='bank')

Step3_4 = Step3_3[Step3_3['correlation'].between(0, 0.6)]
Step3_4


In [0]:
Step3 = Step2[Step2['Symbol'].isin(Step3_4['Symbol'])]
Step3


### Step 4 & 5 — Beta Filter and CAPM Floor

Keeps only stocks with Beta between 0.5 and 1 (moderate market sensitivity), and ensures CAPM expected return exceeds the risk-free rate.


In [0]:
Step4 = Step3[Step3['Beta'].between(0.5, 0.99999)]
Step5 = Step4[Step4['CAPM'] > rf]

# Minimum-variance weighting: weight ∝ 1/σ (lower volatility = higher weight)
Step5 = Step5.copy()
Step5['Weight']      = (1 / Step5['Return_SD']) / (1 / Step5['Return_SD']).sum()
Step5['Port_Return'] = Step5['CAGR'] * Step5['Weight']
Step5


In [0]:
spark_fp = spark.createDataFrame(Step5)
spark_fp.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.V2_Final_PortFolio')


## 11 · Portfolio-Level Analytics

Calculates aggregate portfolio metrics using the selected stocks and their weights.

| Metric | Meaning |
|---|---|
| **Covariance Matrix** | Annualised pairwise covariance of returns — core input to portfolio variance |
| **Portfolio SD** | Overall portfolio volatility accounting for diversification: `√(wᵀ × Σ × w)` |
| **Portfolio Beta** | Weighted average Beta — measures aggregate market sensitivity |
| **Rho (ρ)** | Correlation of portfolio with the market: `(β × Market_SD) / Portfolio_SD` |


In [0]:
# Covariance matrix (annualised)
Step6_1    = mf[mf['Symbol'].isin(Step5['Symbol'])].pivot(index='Date', columns='Symbol', values='Stock_Return')
Cov_Matrix = Step6_1.cov() * 252

# Melt for Power BI
Step6_2 = Cov_Matrix.reset_index()
Step6_3 = Step6_2.melt(id_vars='Symbol', value_name='correlation', var_name='bank')
Step6_3


In [0]:
spark_pcm = spark.createDataFrame(Step6_3)
spark_pcm.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.V2_Portfolio_CovMatrix')


In [0]:
weights_aligned  = Step5.set_index('Symbol')['Weight']
Cov_Matrix_idx   = Cov_Matrix  # already computed above, index preserved

# 1. Portfolio SD: sqrt(w' * Σ * w)
Port_SD   = np.sqrt(weights_aligned.T.dot(Cov_Matrix_idx).dot(weights_aligned))

# 2. Portfolio Beta: weighted average of individual betas
Port_Beta = (weights_aligned * Step5.set_index('Symbol')['Beta']).sum()

# 3. Market SD
Market_SD_val = Step5['Market_SD'].iloc[0]

# 4. Rho — correlation of portfolio with market
Rho = (Port_Beta * Market_SD_val) / Port_SD

PortFolio_Report = pd.DataFrame({
    'Portfolio': ['Portfolio'],
    'Beta'     : [round(float(Port_Beta), 4)],
    'SD'       : [round(float(Port_SD),   4)],
    'Rho'      : [round(float(Rho),       4)]
})
PortFolio_Report


In [0]:
spark_pr = spark.createDataFrame(PortFolio_Report)
spark_pr.write \
    .format('delta') \
    .mode('overwrite') \
    .option('overwriteSchema', 'true') \
    .saveAsTable('dbo.V2_Portfolio_Report')


## 12 · Efficient Frontier

Sweeps all possible weight combinations between the two final portfolio stocks (from -30% to +130%, allowing for short positions) and plots the resulting risk-return space.

**Why this is done even though weights are already optimised:**  
The minimum-variance weights in Step 5 are already optimal for CU savings.  
This sweep is purely for **Power BI visualisation** — it shows the full frontier curve so viewers can see *where* the chosen portfolio sits on the risk-return spectrum.

> **Bug fix applied:** `Cov_Matrix` is used directly here (with its original index) instead of `Step6_2` which had its index modified in the previous cell. This prevents the `NameError: name 'Step6_2' is not defined` error when cells are run independently.


In [0]:
final_stocks = list(Step5['Symbol'])

Port_Statistic = pd.DataFrame({
    final_stocks[0]: np.arange(-0.30, 1.30, 0.05).round(4)
})
Port_Statistic[final_stocks[1]] = 1 - Port_Statistic[final_stocks[0]]

# 1. Portfolio SD at each weight combination
Port_Statistic['SD'] = np.sqrt(
    (Port_Statistic[final_stocks].dot(Cov_Matrix_idx) * Port_Statistic[final_stocks]).sum(axis=1)
)

# 2. Portfolio Return at each weight combination
cagr_series             = Step5.set_index('Symbol')['CAGR']
Port_Statistic['Return']        = Port_Statistic[final_stocks].dot(cagr_series)

# 3. Sharpe Ratio at each weight combination
Port_Statistic['Sharpe_Ratio']  = (Port_Statistic['Return'] - rf) / Port_Statistic['SD']

display(Port_Statistic)


In [0]:
spark_PS = spark.createDataFrame(Port_Statistic)
spark_PS.write \
    .format('delta') \
    .mode('overwrite') \
    .saveAsTable('dbo.V2_Efficient_Frontier')
